# Lennard--Jones and Morse radial pair potentials

## Prerequisites

See the [radial pair-potential lecture note](../../../docs/lecturenotes/solidstate/interatomic-potentials/radial-pair-potentials/index.md). This laboratory assumes conservative radial forces, potential-energy derivatives, and local harmonic curvature.

## Objectives

1. Evaluate unit-aware Lennard--Jones 12-6 and Morse pair potentials.
2. Match their well depth, equilibrium distance, and curvature at the minimum.
3. Verify the signed radial-force convention $F_r=-dV/dr$ numerically.
4. Compare near-equilibrium agreement with their different large-distance tails.

All parameters are illustrative synthetic values, not a fit to a particular material.


## Models

The Lennard--Jones 12-6 potential is

$$
V_{\mathrm{LJ}}(r)=4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^6\right],
$$

with equilibrium distance $r_0=2^{1/6}\sigma$, minimum $-\epsilon$, and curvature

$$
V_{\mathrm{LJ}}''(r_0)=\frac{72\epsilon}{r_0^2}.
$$

The shifted Morse potential is

$$
V_{\mathrm M}(r)=D\left[\left(1-e^{-a(r-r_e)}\right)^2-1\right],
$$

with minimum $-D$ and curvature $V_{\mathrm M}''(r_e)=2Da^2$. Setting
$D=\epsilon$, $r_e=r_0$, and $a=6/r_0$ matches the minima and local curvature.
For both models, positive radial force is repulsive and

$$
F_r(r)=-\frac{dV}{dr}.
$$


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.solidstate.interatomic_potentials.pair.lennard_jones_12_6.parameters import (  # noqa: E501
    LennardJonesPotentialParameters,
)
from projectkoios.physkit.solidstate.interatomic_potentials.pair.lennard_jones_12_6.potential import (  # noqa: E501
    LennardJonesPotential,
)
from projectkoios.physkit.solidstate.interatomic_potentials.pair.morse.parameters import (  # noqa: E501
    MorsePotentialParameters,
)
from projectkoios.physkit.solidstate.interatomic_potentials.pair.morse.potential import (  # noqa: E501
    MorsePotential,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

In [ ]:
energy_unit = PhysicalUnit(expression="electron_volt")
length_unit = PhysicalUnit(expression="angstrom")
force_unit = PhysicalUnit(expression="electron_volt / angstrom")
well_depth = 0.01
zero_crossing_distance = 3.4
lennard_jones_model = LennardJonesPotential(
    parameters=LennardJonesPotentialParameters(
        well_depth=ScalarQuantity(magnitude=well_depth, unit=energy_unit),
        zero_crossing_distance=ScalarQuantity(
            magnitude=zero_crossing_distance,
            unit=length_unit,
        ),
    )
)
equilibrium_distance = lennard_jones_model.equilibrium_distance.magnitude
matched_inverse_range = 6.0 / equilibrium_distance
morse_model = MorsePotential(
    parameters=MorsePotentialParameters(
        dissociation_energy=ScalarQuantity(
            magnitude=well_depth,
            unit=energy_unit,
        ),
        inverse_range=ScalarQuantity(
            magnitude=matched_inverse_range,
            unit=PhysicalUnit(expression="1 / angstrom"),
        ),
        equilibrium_distance=ScalarQuantity(
            magnitude=equilibrium_distance,
            unit=length_unit,
        ),
    )
)
distances = VectorQuantity(
    magnitude=np.linspace(
        0.9 * equilibrium_distance,
        2.5 * equilibrium_distance,
        600,
        dtype=np.float64,
    ),
    unit=length_unit,
)
lennard_jones = lennard_jones_model.evaluate(
    distances=distances,
    energy_unit=energy_unit,
    force_unit=force_unit,
)
morse = morse_model.evaluate(
    distances=distances,
    energy_unit=energy_unit,
    force_unit=force_unit,
)

In [ ]:
equilibrium_distances = VectorQuantity(
    magnitude=np.asarray([equilibrium_distance], dtype=np.float64),
    unit=length_unit,
)
lennard_jones_equilibrium = lennard_jones_model.evaluate(
    distances=equilibrium_distances,
    energy_unit=energy_unit,
    force_unit=force_unit,
)
morse_equilibrium = morse_model.evaluate(
    distances=equilibrium_distances,
    energy_unit=energy_unit,
    force_unit=force_unit,
)
assert np.isclose(
    lennard_jones_equilibrium.potential_energies.magnitude[0], -well_depth
)
assert np.isclose(morse_equilibrium.potential_energies.magnitude[0], -well_depth)
assert np.isclose(
    lennard_jones_equilibrium.radial_forces.magnitude[0], 0.0, atol=1.0e-14
)
assert np.isclose(morse_equilibrium.radial_forces.magnitude[0], 0.0, atol=1.0e-14)

numerical_lennard_jones_force = -np.gradient(
    lennard_jones.potential_energies.magnitude,
    distances.magnitude,
)
numerical_morse_force = -np.gradient(
    morse.potential_energies.magnitude,
    distances.magnitude,
)
interior = slice(2, -2)
lj_force_error = float(
    np.max(
        np.abs(
            numerical_lennard_jones_force[interior]
            - lennard_jones.radial_forces.magnitude[interior]
        )
    )
    / np.max(np.abs(lennard_jones.radial_forces.magnitude[interior]))
)
morse_force_error = float(
    np.max(
        np.abs(
            numerical_morse_force[interior] - morse.radial_forces.magnitude[interior]
        )
    )
    / np.max(np.abs(morse.radial_forces.magnitude[interior]))
)
assert lj_force_error < 2.0e-3
assert morse_force_error < 2.0e-3
lj_force_error, morse_force_error

In [ ]:
figure, (energy_axis, force_axis) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(9.0, 8.0),
    sharex=True,
)
energy_axis.plot(
    distances.magnitude,
    lennard_jones.potential_energies.magnitude,
    label="Lennard–Jones 12-6",
)
energy_axis.plot(
    distances.magnitude,
    morse.potential_energies.magnitude,
    label="curvature-matched Morse",
)
energy_axis.axhline(0.0, color="black", linewidth=0.7)
energy_axis.axvline(
    equilibrium_distance,
    color="black",
    linestyle="--",
    label="matched equilibrium distance",
)
energy_axis.set(
    ylabel="potential energy (eV)",
    title="Illustrative radial pair energies",
)
energy_axis.legend()
energy_axis.grid(True)

force_axis.plot(
    distances.magnitude,
    lennard_jones.radial_forces.magnitude,
    label="Lennard–Jones radial force",
)
force_axis.plot(
    distances.magnitude,
    morse.radial_forces.magnitude,
    label="Morse radial force",
)
force_axis.axhline(0.0, color="black", linewidth=0.7)
force_axis.axvline(equilibrium_distance, color="black", linestyle="--")
force_axis.set(
    xlabel="separation (Å)",
    ylabel="radial force (eV Å⁻¹)",
    title="Positive repulsion and negative attraction",
)
force_axis.legend()
force_axis.grid(True)
figure.tight_layout()
plt.show()

## Interpretation

Matching the minimum and curvature makes the two potentials locally similar near equilibrium, but it does not make them globally equivalent. The Lennard--Jones attraction has an algebraic $r^{-6}$ tail, whereas the Morse interaction approaches zero exponentially.

The corrected Morse force is positive for compressed separations and negative for stretched separations. The original exploratory notebook used the opposite sign while labeling it $-dV/dr$.

These checks verify the represented mathematics and numerical implementation only. The illustrative parameters are not material validation or an interatomic potential suitable for predictive simulation.

## Exercises

1. Change the Morse inverse range without matching curvature and compare the wells.
2. Verify the analytical Lennard--Jones curvature by centered second differences.
3. Plot the large-distance tails on logarithmic axes.
4. Identify additional evidence required before using either model for a specific material.
